# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aamnamalik16-bit/flyrank-ML-internship/blob/main/work/notebooks/w03_data_contract.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os

if not os.path.exists('flyrank-ML-internship'):
    !git clone https://github.com/aamnamalik16-bit/flyrank-ML-internship.git

os.chdir('flyrank-ML-internship')

!pip install duckdb -q

import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")

rel = "hf://datasets/FlyRank/internship-warehouse"
print("Setup complete!")

Cloning into 'flyrank-ML-internship'...
remote: Enumerating objects: 163, done.
remote: Counting objects: 100% (163/163), done.
remote: Compressing objects: 100% (106/106), done.
remote: Total 163 (delta 73), reused 114 (delta 41), pack-reused 0 (from 0)
Receiving objects: 100% (163/163), 1.85 MiB | 5.14 MiB/s, done.
Resolving deltas: 100% (73/73), done.
Setup complete!


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [2]:
print("Unit of analysis: one content item x one query hash")
print("Table: fact_content_query_90d")
print("Time window: fixed 90-day snapshot")
print("Label: is_rising = (clicks_last30 > clicks_prev30)")
print("Excluded: inactive clients (is_active IS NOT TRUE)")

Unit of analysis: one content item x one query hash
Table: fact_content_query_90d
Time window: fixed 90-day snapshot
Label: is_rising = (clicks_last30 > clicks_prev30)
Excluded: inactive clients (is_active IS NOT TRUE)


One row = one content item paired with one query hash, over a fixed 90-day window.

Table: fact_content_query_90d
Time window: Fixed 90-day snapshot ending at export date (2026-07-03), with clicks_last30 and clicks_prev30 sub-windows already computed
Label / proxy: is_rising = (clicks_last30 > clicks_prev30) — pages where query clicks grew in the last 30 days vs the previous 30 days
Deliberately excluded: clients where is_active IS NOT TRUE — inactive clients have incomplete or unreliable query histories

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [3]:
print("Fields classified into: feature / label / context / excluded")
print("Label source: clicks_last30 and clicks_prev30")
print("Features: impressions, position, query_count, clicks_90d")
print("Context: content_hash_id, client_hash_id, query_hash_id")
print("Excluded: is_active (filter only)")

Fields classified into: feature / label / context / excluded
Label source: clicks_last30 and clicks_prev30
Features: impressions, position, query_count, clicks_90d
Context: content_hash_id, client_hash_id, query_hash_id
Excluded: is_active (filter only)


Feature / label / context / excluded breakdown:

Field	Bucket	Reason
clicks_last30	Label source	Used to compute is_rising — never a feature
clicks_prev30	Label source	Used to compute is_rising — never a feature
impressions_last30	Feature	Knowable at decision time — measures query visibility
impressions_prev30	Feature	Knowable at decision time — measures prior visibility
gsc_avg_position	Feature	Knowable at decision time — average ranking position
query_count	Feature	Knowable at decision time — number of queries per page
clicks_90d	Feature	Knowable at decision time — total 90-day click volume
content_hash_id	Context	ID for joining/grouping only — not a model input
client_hash_id	Context	ID for client-level grouping and splits — not a model input
query_hash_id	Context	ID for query identity — not a model input
is_active	Excluded	Client status flag — used for filtering only, not modeling

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [10]:
import pandas as pd

# Build the feature frame from the query table
df = con.sql(f"""
    SELECT
        content_hash_id,
        client_hash_id,
        query_hash_id,
        impressions_last30,
        impressions_prev30,
        clicks_last30,
        clicks_prev30,
        avg_position_last30,
        (clicks_last30 - clicks_prev30) as click_delta,
        (clicks_last30 > clicks_prev30) as is_rising
    FROM read_parquet('{rel}/fact_content_query_90d.parquet')
    WHERE (impressions_last30 > 0) IS TRUE
""").df()

print("Feature frame shape:", df.shape)
df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature frame shape: (1883490, 10)


,content_hash_id,client_hash_id,query_hash_id,impressions_last30,impressions_prev30,clicks_last30,clicks_prev30,avg_position_last30,click_delta,is_rising
0,content_447894f2faf0d2bc,client_08a6a72ff48e62c0,query_9f0c36a6ae2a6a99,11,5,0,0,24.272727,0,False
1,content_447894f2faf0d2bc,client_08a6a72ff48e62c0,query_a032820b5467e996,1,1,0,0,13.000000,0,False
2,content_447894f2faf0d2bc,client_08a6a72ff48e62c0,query_bf21a07e311fed62,3,3,0,0,5.000000,0,False
3,content_447894f2faf0d2bc,client_08a6a72ff48e62c0,query_c9de7d945a73e134,5,6,0,0,18.600000,0,False
4,content_447b93d5ff670356,client_08a6a72ff48e62c0,query_9dfad337ed50a6f9,1,2,0,0,49.000000,0,False


In [5]:
con.sql(f"""
    SELECT content_hash_id, query_hash_id, COUNT(*) as c
    FROM read_parquet('{rel}/fact_content_query_90d.parquet')
    GROUP BY content_hash_id, query_hash_id
    HAVING c > 1
    LIMIT 5
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────────┬───────────────┬───────┐
│ content_hash_id │ query_hash_id │   c   │
│     varchar     │    varchar    │ int64 │
├─────────────────┴───────────────┴───────┤
│                 0 rows                  │
└─────────────────────────────────────────┘



In [7]:
con.sql(f"""
    SELECT
        COUNT(*) as total_rows,
        COUNT(DISTINCT client_hash_id) as clients,
        COUNT(DISTINCT content_hash_id) as content_items,
        MIN(window_start) as earliest_date,
        MAX(window_start) as latest_date
    FROM read_parquet('{rel}/fact_content_query_90d.parquet')
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬─────────┬───────────────┬───────────────┬─────────────┐
│ total_rows │ clients │ content_items │ earliest_date │ latest_date │
│   int64    │  int64  │     int64     │     date      │    date     │
├────────────┼─────────┼───────────────┼───────────────┼─────────────┤
│    2414248 │      52 │        133852 │ 2026-04-02    │ 2026-04-02  │
└────────────┴─────────┴───────────────┴───────────────┴─────────────┘



In [8]:
con.sql(f"""
    SELECT
        COUNT(*) as total_rows,
        COUNT(CASE WHEN (impressions_last30 > 0) IS TRUE THEN 1 END) as has_impressions,
        COUNT(CASE WHEN (clicks_last30 > 0) IS TRUE THEN 1 END) as has_clicks
    FROM read_parquet('{rel}/fact_content_query_90d.parquet')
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬─────────────────┬────────────┐
│ total_rows │ has_impressions │ has_clicks │
│   int64    │      int64      │   int64    │
├────────────┼─────────────────┼────────────┤
│    2414248 │         1883490 │      85115 │
└────────────┴─────────────────┴────────────┘



Verification results:

Query 1 — Grain check: 0 duplicate rows returned — confirmed that one row = one content item × one query hash. The grain holds.

Query 2 — Row count and date span: 2,414,248 total rows across 52 clients and 133,852 content items. Window start date is 2026-04-02 (fixed 90-day snapshot).

Query 3 — Availability (checked with IS TRUE): Out of 2,414,248 total rows, 1,883,490 rows have impressions in the last 30 days, and only 85,115 rows have clicks in the last 30 days. This means clicks are sparse — a minimum impressions filter will be needed before modeling.

Paste that in the markdown cell. Done?




## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [9]:
print("Named limitation: fixed 90-day window cannot be re-sliced by month.")
print("clicks_last30 vs clicks_prev30 are pre-computed — no custom time windows.")
print("Future work: join to fact_content_daily_performance for flexible windows.")

Named limitation: fixed 90-day window cannot be re-sliced by month.
clicks_last30 vs clicks_prev30 are pre-computed — no custom time windows.
Future work: join to fact_content_daily_performance for flexible windows.


One named limitation of this slice:

The fact_content_query_90d table uses a fixed 90-day window — it does not partition by month like the daily performance table. This means I cannot safely develop label logic on a mid-panel month and then test on a later month, because the window overlaps across all rows. The click sub-windows (clicks_last30 vs clicks_prev30) are the only time separation available, and they are already baked in — I cannot redefine them. This limits my ability to build a true past→future label without joining to the daily performance table.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.